# moodcompiler — MentalBERT Fine-Tuning on Google Colab
**"Behind Every Signal, There's a Story."**

This notebook fine-tunes `mental/mental-bert-base-uncased` for 4-class depression severity classification:
- **0: Normal**
- **1: Mild**
- **2: Moderate**
- **3: Severe**

### Instructions:
1. Go to **Runtime -> Change runtime type -> T4 GPU**.
2. Run all cells in sequence.
3. Download `mentalbert_depression.zip` and extract into `moodcompiler/backend/models/`.

In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Install tooling
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
# 3. Upload Reddit_depression_dataset.csv
from google.colab import files
import os

if not os.path.exists('Reddit_depression_dataset.csv'):
    print('Please upload Reddit_depression_dataset.csv from backend/data/reddit/ :')
    uploaded = files.upload()
else:
    print('Dataset already found!')

In [ ]:
# 4. Fine-Tune MentalBERT with Stratified Split & Balanced Loss
import re, json, torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

MODEL_NAME = 'mental/mental-bert-base-uncased'
OUTPUT_DIR = 'mentalbert_depression'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 3
LR = 2e-5

LABEL_MAP = {'minimum': 'Normal', 'mild': 'Mild', 'moderate': 'Moderate', 'severe': 'Severe'}
LABEL2ID = {'Normal': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3}
ID2LABEL = {0: 'Normal', 1: 'Mild', 2: 'Moderate', 3: 'Severe'}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv('Reddit_depression_dataset.csv').dropna(subset=['text', 'label']).copy()
df['label_std'] = df['label'].str.lower().str.strip().map(LABEL_MAP)
df = df.dropna(subset=['label_std']).copy()
df['clean'] = df['text'].apply(clean_text)
df['target'] = df['label_std'].map(LABEL2ID)

X_train, X_val, y_train, y_val = train_test_split(
    df['clean'].values, df['target'].values, test_size=0.20, random_state=42, stratify=df['target'].values
)

class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1, 2, 3]), y=y_train)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(DepressionDataset(X_train, y_train, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(DepressionDataset(X_val, y_val, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID)
model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

best_f1 = 0.0
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
            val_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            val_targets.extend(batch['labels'].cpu().numpy())

    acc = accuracy_score(val_targets, val_preds)
    f1 = f1_score(val_targets, val_preds, average='macro')
    print(f'Epoch {epoch+1}/{EPOCHS} -> Val Accuracy: {acc*100:.2f}% | Macro F1: {f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
            json.dump({'model': MODEL_NAME, 'val_acc': round(acc*100, 2), 'val_f1': round(f1, 4)}, f)

print('\n[DONE] Best Model Saved to ' + OUTPUT_DIR)

In [ ]:
# 5. Compress & Download Trained Model
!zip -r mentalbert_depression.zip mentalbert_depression
from google.colab import files
files.download('mentalbert_depression.zip')
print('Model downloaded successfully!')